# Simulação térmica por diferenças finitas — Gêmeo Digital da GPU

**Projeto:** Gêmeo Digital Térmico de Hardware Computacional  
**Aplicação:** placa GPU de inferência de IA em operação contínua  
**Referência térmica:** NVIDIA L4 — potência de projeto de 72 W  
**Material do dissipador:** Alumínio 6061  
**Método:** equação do calor 2D em regime permanente, diferenças finitas e fronteira convectiva.

## Objetivo

Resolver numericamente a distribuição de temperatura no dissipador do CAD integrado ao projeto e verificar a temperatura de equilíbrio para diferentes coeficientes de convecção `h`.

O modelo segue os parâmetros definidos no relatório:

- base: **170 × 70 × 5 mm**;
- 12 aletas longitudinais;
- aletas: **170 × 2 × 25 mm**;
- altura total: **30 mm**;
- potência térmica: **72 W**;
- ambiente nominal: **25 °C**;
- limite de projeto: **80 °C**;
- condutividade térmica adotada para Al 6061: **160 W/m·K**.

> **Nota sobre o CoolProp:** CoolProp é utilizado para propriedades do fluido (ar). As propriedades do sólido Al 6061 são parâmetros do próprio projeto, pois CoolProp não é uma biblioteca geral de propriedades de sólidos metálicos. O notebook não depende de um solver externo. Se o ambiente já possuir CoolProp, ele será utilizado; caso contrário, o notebook usa um fallback documentado para manter a execução sem instalação.

In [ ]:
# Bibliotecas — apenas NumPy e Matplotlib são necessárias para a solução numérica.
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# CoolProp é opcional no ambiente de execução.
# Se estiver disponível, será usado para propriedades do ar.
try:
    from CoolProp.CoolProp import PropsSI
    COOLPROP_OK = True
except ImportError:
    COOLPROP_OK = False

print("CoolProp disponível:", COOLPROP_OK)
print("NumPy:", np.__version__)

## 1. Parâmetros geométricos e térmicos

Os valores abaixo são os mesmos utilizados na especificação CAD integrada. O espaçamento entre aletas é calculado automaticamente:

\[
g = \frac{W-Nt_f}{N+1}
\]

onde `W` é a largura da base, `N` o número de aletas e `t_f` a espessura de cada aleta.

In [ ]:
# =========================
# PARÂMETROS DO GÊMEO DIGITAL
# =========================

# Geometria [m]
W = 70e-3          # largura da base
L = 170e-3         # profundidade/comprimento das aletas
t_base = 5e-3      # espessura da base
h_fin = 25e-3      # altura das aletas
N_fin = 12         # número de aletas
t_fin = 2e-3       # espessura de cada aleta

H_total = t_base + h_fin
gap = (W - N_fin*t_fin) / (N_fin + 1)

# Condições térmicas
P = 72.0           # W
T_amb_C = 25.0     # °C
T_lim_C = 80.0     # °C

# Material — Al 6061, conforme o relatório
k_al = 160.0       # W/(m.K)
rho_al = 2700.0    # kg/m³
cp_al = 896.0      # J/(kg.K), usado apenas se for feita análise transiente

print(f"Dimensão externa: {W*1e3:.1f} x {L*1e3:.1f} x {H_total*1e3:.1f} mm")
print(f"Espaçamento entre aletas: {gap*1e3:.3f} mm")
print(f"Potência térmica: {P:.1f} W")
print(f"Temperatura ambiente: {T_amb_C:.1f} °C")
print(f"Limite de projeto: {T_lim_C:.1f} °C")

## 2. Propriedades do ar com CoolProp

Para uma simulação com convecção, o coeficiente `h` é fornecido como condição de contorno. O CoolProp é usado para obter propriedades do ar na temperatura de filme.

A densidade e a viscosidade não entram diretamente na equação estacionária quando `h` é imposto, mas são registradas para tornar o modelo reproduzível e permitir uma futura correlação de convecção.

Se CoolProp não estiver disponível no ambiente, são usados valores de referência próximos às condições ambientes, evitando que a execução seja interrompida.

In [ ]:
# =========================
# PROPRIEDADES DO AR
# =========================

Tfilm_K = T_amb_C + 273.15

if COOLPROP_OK:
    rho_air = PropsSI("D", "T", Tfilm_K, "P", 101325, "Air")
    mu_air = PropsSI("V", "T", Tfilm_K, "P", 101325, "Air")
    k_air = PropsSI("L", "T", Tfilm_K, "P", 101325, "Air")
    cp_air = PropsSI("C", "T", Tfilm_K, "P", 101325, "Air")
    source_air = "CoolProp"
else:
    # Fallback para manter o notebook executável sem instalação.
    rho_air = 1.184
    mu_air = 1.849e-5
    k_air = 0.0255
    cp_air = 1007.0
    source_air = "fallback de referência"

print(f"Fonte das propriedades: {source_air}")
print(f"rho_ar = {rho_air:.4f} kg/m³")
print(f"mu_ar  = {mu_air:.4e} Pa.s")
print(f"k_ar   = {k_air:.5f} W/(m.K)")
print(f"cp_ar  = {cp_air:.1f} J/(kg.K)")

## 3. Domínio 2D do dissipador

O modelo usa uma seção transversal `x-y` da base e das 12 aletas. A profundidade de 170 mm entra no balanço energético para converter o problema 2D em uma representação do componente extrudado.

A região sólida é definida por:

- base: `0 ≤ y ≤ 5 mm`;
- aletas: `5 mm < y ≤ 30 mm`, nas 12 posições.

A face inferior da base representa o contato com a GPU/TIM e, portanto, **não é tratada como uma superfície convectiva**. O calor de 72 W é introduzido na base. As demais superfícies expostas ao ar recebem condição de convecção.

### Equação governante

Em regime permanente:

\[
\nabla\cdot(k\nabla T)+\dot q=0
\]

Na superfície exposta:

\[
-k\frac{\partial T}{\partial n}=h(T-T_\infty)
\]

Para o modelo discreto, a condição convectiva é incorporada à equação de cada nó de superfície.

In [ ]:
def build_geometry(nx=101, ny=41):
    x = np.linspace(0.0, W, nx)
    y = np.linspace(0.0, H_total, ny)
    dx = x[1] - x[0]
    dy = y[1] - y[0]
    X, Y = np.meshgrid(x, y)

    solid = Y <= t_base + 1e-12

    # Posicionamento uniforme das 12 aletas.
    for i in range(N_fin):
        x0 = gap + i*(t_fin + gap)
        x1 = x0 + t_fin
        solid |= (
            (Y >= t_base - 1e-12) &
            (Y <= H_total + 1e-12) &
            (X >= x0 - 1e-12) &
            (X <= x1 + 1e-12)
        )

    return x, y, X, Y, solid, dx, dy


x, y, X, Y, solid, dx, dy = build_geometry()

plt.figure(figsize=(10, 4))
plt.imshow(
    solid,
    origin="lower",
    extent=[0, W*1e3, 0, H_total*1e3],
    aspect="equal",
    interpolation="nearest"
)
plt.xlabel("Largura x [mm]")
plt.ylabel("Altura y [mm]")
plt.title("Domínio 2D discretizado do dissipador")
plt.colorbar(label="Sólido = 1")
plt.show()

print(f"Malha: {len(x)} x {len(y)} nós")
print(f"dx = {dx*1e3:.3f} mm | dy = {dy*1e3:.3f} mm")

## 4. Discretização por diferenças finitas

Para um nó interno, a forma central da equação é:

\[
\frac{T_E-2T_P+T_W}{\Delta x^2}
+
\frac{T_N-2T_P+T_S}{\Delta y^2}
+
\frac{\dot q}{k}=0
\]

Nas faces expostas, a condição de Robin é incorporada como um termo convectivo. O solucionador abaixo utiliza **Gauss-Seidel com sobre-relaxação (SOR)** em uma malha de diferenças finitas.

O calor volumétrico equivalente na base é:

\[
\dot q = \frac{P}{W\,t_{base}\,L}
\]

Este é um modelo estacionário; portanto, a temperatura obtida após convergência representa a **temperatura de equilíbrio térmico**.

In [ ]:
def solve_steady_state(h_conv, nx=101, ny=41, max_iter=12000,
                       tol=1e-5, omega=1.6, T_amb=T_amb_C):
    """Resolve o campo 2D de temperatura por diferenças finitas.

    h_conv : coeficiente convectivo [W/(m².K)]
    Retorna T em °C e informações de convergência.
    """
    x, y, X, Y, solid, dx, dy = build_geometry(nx, ny)

    T = np.full((ny, nx), T_amb, dtype=float)
    qdot = P / (W * t_base * L)  # W/m³

    # Coeficientes diagonais.
    den = np.zeros((ny, nx), dtype=float)
    ambient_term = np.zeros((ny, nx), dtype=float)

    directions = [
        (-1, 0, dx, False),  # esquerda
        ( 1, 0, dx, False),  # direita
        ( 0,-1, dy, True),   # baixo
        ( 0, 1, dy, False),  # cima
    ]

    for di, dj, d, is_bottom in directions:
        neighbor = np.zeros_like(solid)

        if di == -1:
            neighbor[:, 1:] = solid[:, :-1]
        elif di == 1:
            neighbor[:, :-1] = solid[:, 1:]
        elif dj == -1:
            neighbor[1:, :] = solid[:-1, :]
        else:
            neighbor[:-1, :] = solid[1:, :]

        # Condução entre dois nós sólidos.
        c_cond = k_al / d**2
        den += solid * neighbor * c_cond

        # Face exposta.
        exposed = solid & (~neighbor)

        # Fundo: contato com GPU/TIM, sem convecção neste modelo.
        if is_bottom:
            continue

        # Robin: -k dT/dn = h(T-Tinf)
        # Com célula de centro na superfície: distância até a face = d/2.
        c_conv = 2.0 * h_conv / d
        den += exposed * c_conv
        ambient_term += exposed * c_conv * T_amb

    # Nós de borda sem coeficiente não devem ser atualizados.
    active = solid & (den > 0)

    # Máscaras de xadrez para Gauss-Seidel vermelho/preto.
    jj, ii = np.indices((ny, nx))
    colors = (ii + jj) & 1

    for iteration in range(1, max_iter + 1):
        T_old = T.copy()

        for color in (0, 1):
            # Numerador: ambiente + calor gerado + vizinhos sólidos.
            num = ambient_term + qdot * solid

            # Esquerda
            nb = np.zeros_like(T)
            nb[:, 1:] = T[:, :-1]
            valid = np.zeros_like(solid)
            valid[:, 1:] = solid[:, :-1]
            num += valid * (k_al/dx**2) * nb

            # Direita
            nb = np.zeros_like(T)
            nb[:, :-1] = T[:, 1:]
            valid = np.zeros_like(solid)
            valid[:, :-1] = solid[:, 1:]
            num += valid * (k_al/dx**2) * nb

            # Baixo
            nb = np.zeros_like(T)
            nb[1:, :] = T[:-1, :]
            valid = np.zeros_like(solid)
            valid[1:, :] = solid[:-1, :]
            num += valid * (k_al/dy**2) * nb

            # Cima
            nb = np.zeros_like(T)
            nb[:-1, :] = T[1:, :]
            valid = np.zeros_like(solid)
            valid[:-1, :] = solid[1:, :]
            num += valid * (k_al/dy**2) * nb

            candidate = np.divide(num, den, out=T.copy(), where=active)
            select = active & (colors == color)
            T[select] = (1.0 - omega)*T[select] + omega*candidate[select]

        error = np.max(np.abs(T[solid] - T_old[solid]))

        if error < tol:
            break

    T[~solid] = np.nan

    return {
        "x": x, "y": y, "X": X, "Y": Y, "solid": solid,
        "T": T, "iterations": iteration, "error": error,
        "dx": dx, "dy": dy, "qdot": qdot
    }


# Teste no caso nominal do projeto.
result_nominal = solve_steady_state(h_conv=40.0)
T_nominal = result_nominal["T"]

print(f"Iterações: {result_nominal['iterations']}")
print(f"Erro máximo final: {result_nominal['error']:.3e} °C")
print(f"Tmin = {np.nanmin(T_nominal):.2f} °C")
print(f"Tmax = {np.nanmax(T_nominal):.2f} °C")
print(f"Margem até 80 °C = {T_lim_C - np.nanmax(T_nominal):.2f} °C")

In [ ]:
# =========================
# CAMPO DE TEMPERATURA
# =========================

plt.figure(figsize=(11, 5))
im = plt.imshow(
    T_nominal,
    origin="lower",
    extent=[0, W*1e3, 0, H_total*1e3],
    aspect="equal",
    interpolation="bilinear"
)
plt.xlabel("Largura x [mm]")
plt.ylabel("Altura y [mm]")
plt.title("Temperatura de equilíbrio — h = 40 W/m²·K")
plt.colorbar(im, label="Temperatura [°C]")
plt.contour(
    result_nominal["X"]*1e3,
    result_nominal["Y"]*1e3,
    result_nominal["solid"].astype(float),
    levels=[0.5],
    linewidths=0.8
)
plt.show()

## 5. Estudo de sensibilidade ao coeficiente convectivo

O relatório do projeto avalia:

`h = 5, 8, 10, 15, 20, 30 e 40 W/m²·K`.

O objetivo é verificar numericamente quando o dissipador ultrapassa ou permanece abaixo do limite de projeto de **80 °C**.

A classificação é feita exclusivamente pelo critério térmico:

- **Atende:** `Tmax ≤ 80 °C`;
- **Não atende:** `Tmax > 80 °C`.

Os resultados do notebook representam o **modelo de diferenças finitas integrado ao CAD**, portanto podem diferir dos números do modelo simplificado anterior do relatório. Essa diferença é importante e deve ser documentada, não escondida.

In [ ]:
h_values = [5, 8, 10, 15, 20, 30, 40]
results = []

for h in h_values:
    r = solve_steady_state(h_conv=h)
    Tmax = np.nanmax(r["T"])
    Tmin = np.nanmin(r["T"])
    results.append({
        "h_W_m2K": h,
        "Tmax_C": Tmax,
        "Tmin_C": Tmin,
        "margin_C": T_lim_C - Tmax,
        "status": "Atende" if Tmax <= T_lim_C else "Não atende",
        "iterations": r["iterations"],
        "error_C": r["error"]
    })

print(" h [W/m²K] | Tmin [°C] | Tmax [°C] | Margem [°C] | Critério")
print("-"*67)
for r in results:
    print(f"{r['h_W_m2K']:10.0f} | {r['Tmin_C']:10.2f} | {r['Tmax_C']:10.2f} | "
          f"{r['margin_C']:11.2f} | {r['status']}")

In [ ]:
# Curva de sensibilidade
hs = np.array([r["h_W_m2K"] for r in results])
Tmaxs = np.array([r["Tmax_C"] for r in results])

plt.figure(figsize=(8, 5))
plt.plot(hs, Tmaxs, "o-", label="Diferenças finitas")
plt.axhline(T_lim_C, linestyle="--", label="Limite de projeto: 80 °C")
plt.xlabel("Coeficiente convectivo h [W/m²·K]")
plt.ylabel("Temperatura máxima de equilíbrio [°C]")
plt.title("Sensibilidade da temperatura máxima à convecção")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 6. Verificação do balanço energético

Em equilíbrio térmico, a potência recebida pelo sólido deve ser aproximadamente igual à potência removida por convecção:

\[
P_{conv} = \int_A h(T-T_\infty)\,dA
\]

O cálculo abaixo estima a potência convectiva pelas faces expostas da malha. A diferença relativa entre `P_conv` e os 72 W fornece uma verificação adicional da convergência e da consistência do modelo.

In [ ]:
def convection_power(result, h_conv, T_amb=T_amb_C):
    T = result["T"]
    solid = result["solid"]
    dx, dy = result["dx"], result["dy"]

    # Integração aproximada por faces expostas.
    Pconv = 0.0

    directions = [
        (-1, 0, dx, False),
        ( 1, 0, dx, False),
        ( 0,-1, dy, True),
        ( 0, 1, dy, False),
    ]

    for di, dj, d, is_bottom in directions:
        neighbor = np.zeros_like(solid)
        if di == -1:
            neighbor[:, 1:] = solid[:, :-1]
        elif di == 1:
            neighbor[:, :-1] = solid[:, 1:]
        elif dj == -1:
            neighbor[1:, :] = solid[:-1, :]
        else:
            neighbor[:-1, :] = solid[1:, :]

        exposed = solid & (~neighbor)

        if is_bottom:
            continue

        # Área da face representada pelo nó: aproximadamente d_other * L.
        face_area = (dy if di != 0 else dx) * L
        Pconv += np.nansum(
            h_conv * (T - T_amb) * exposed
        ) * face_area

    return Pconv


Pconv_nominal = convection_power(result_nominal, 40.0)
balance_error = 100.0 * abs(Pconv_nominal - P) / P

print(f"Potência gerada:      {P:.3f} W")
print(f"Potência convectiva:  {Pconv_nominal:.3f} W")
print(f"Erro de balanço:      {balance_error:.2f} %")

## 7. Temperatura de equilíbrio e comparação com o relatório anterior

O relatório anterior apresentava, para o modelo térmico simplificado:

- `h = 40 W/m²·K` → `Tmax ≈ 41,7 °C`;
- `h = 10 W/m²·K` → `Tmax ≈ 84,2 °C`.

A simulação atual é mais explicitamente ligada à geometria das 12 aletas e às condições de contorno do CAD. Portanto, o resultado numérico atual deve ser tratado como a **versão de referência do gêmeo digital CAD–térmico**.

Se houver diferença entre os resultados, isso não significa automaticamente erro: significa que os modelos matemáticos/geométricos são diferentes. Para a entrega, deve-se informar qual modelo está sendo considerado como versão oficial.

In [ ]:
# Comparação explícita com os valores publicados no relatório anterior.
report_previous = {
    40: 41.7,
    10: 84.2
}

current = {r["h_W_m2K"]: r["Tmax_C"] for r in results}

print("Comparação:")
print("h [W/m²K] | Modelo anterior [°C] | Modelo CAD-FD [°C] | Diferença [°C]")
print("-"*70)

for h in [10, 40]:
    diff = current[h] - report_previous[h]
    print(f"{h:10.0f} | {report_previous[h]:20.2f} | "
          f"{current[h]:18.2f} | {diff:14.2f}")

## 8. Critério de aceitação

Para o projeto, a condição nominal de ventilação forçada é considerada termicamente adequada quando:

\[
T_{max} \leq 80\ ^\circ C
\]

Além da temperatura, o projeto deve considerar a continuidade do fluxo de ar. Uma falha de ventilação reduz `h` e pode elevar rapidamente a temperatura.

O notebook permite alterar:

- potência da GPU;
- temperatura ambiente;
- número e espessura das aletas;
- dimensões da base;
- condutividade do material;
- coeficiente convectivo;
- resolução da malha.

Isso permite utilizar o mesmo código como núcleo computacional do gêmeo digital.

In [ ]:
# =========================
# CENÁRIO NOMINAL FINAL
# =========================
nominal = next(r for r in results if r["h_W_m2K"] == 40)

print("=== VERIFICAÇÃO FINAL ===")
print(f"Potência:              {P:.1f} W")
print(f"Ambiente:              {T_amb_C:.1f} °C")
print(f"h nominal:             40 W/m²·K")
print(f"Tmax de equilíbrio:    {nominal['Tmax_C']:.2f} °C")
print(f"Tmin de equilíbrio:    {nominal['Tmin_C']:.2f} °C")
print(f"Limite de projeto:     {T_lim_C:.1f} °C")
print(f"Margem térmica:        {nominal['margin_C']:.2f} °C")
print(f"Critério:              {nominal['status']}")
print(f"Iterações:             {nominal['iterations']}")
print(f"Erro de convergência:  {nominal['error_C']:.3e} °C")

## 9. Conclusão da simulação

A solução por diferenças finitas resolve a equação do calor no domínio sólido do dissipador e aplica uma condição convectiva nas superfícies expostas.

O valor de `Tmax` após a convergência representa a **temperatura de equilíbrio** para o cenário analisado.

### Integração com o CAD

A correspondência dos principais parâmetros é:

| Parâmetro | CAD / relatório | Notebook |
|---|---|---|
| Base | 170 × 70 × 5 mm | `W`, `L`, `t_base` |
| Aletas | 12 unidades | `N_fin` |
| Espessura | 2 mm | `t_fin` |
| Altura | 25 mm | `h_fin` |
| Material | Al 6061 | `k_al` |
| Potência | 72 W | `P` |
| Ambiente | 25 °C | `T_amb_C` |
| Convecção | 5–40 W/m²·K | `h_values` |
| Limite | 80 °C | `T_lim_C` |

**Regra de rastreabilidade:** se uma dimensão do CAD for alterada, o parâmetro correspondente do notebook deve ser alterado antes de considerar os resultados como pertencentes à mesma versão do gêmeo digital.

In [ ]:
# Opcional: salvar os resultados em CSV para anexar ao repositório do projeto.
import csv

csv_path = Path("resultados_simulacao_cad_fd.csv")
with csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

print(f"Arquivo gerado: {csv_path.resolve()}")